# 01 — Data inspection and initial baseline

Prepare provisional labels from synthetic data and train the original 12-category Logistic Regression baseline. The final seven-category workflow is in notebook 02. Execute cells in order. Recovery steps are in `notebooks/03_model_recovery.ipynb`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Find the project whether this notebook runs from root or notebooks/.
PROJECT_PATH = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "requirements.txt").exists() and (p / "src").is_dir()
)
if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))


In [ ]:
data_path = PROJECT_PATH / "data/raw/transactions-synthetic.csv"
df = pd.read_csv(data_path)
df.head()


In [ ]:
print("Rows and columns:",df.shape)

In [ ]:
print("Column names:",df.columns.tolist());

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("\nCategory counts:")
print(df["category"].value_counts())

print("\nExact dublicate rows:", df.duplicated().sum())

In [ ]:
clean_df = df.drop_duplicates().copy()

In [ ]:
print("Original rows",len(df))
print("After removing dublicates:", len(clean_df))
print("\nCategory counts:")
print(clean_df["category"].value_counts())

In [ ]:
category_mapping = {
    "Restaurants": "Food",
    "Groceries": "Food",
    "Transportation": "Transport",
    "Utilities": "Bills",
    "Rent": "Housing",
    "Mortgage": "Housing",
    "Shopping": "Shopping",
    "Healthcare": "Health",
    "Entertainment": "Entertainment",
    "Personal Care": "Personal Care",
    "Education": "Education",
    "Insurance": "Insurance",
    "Fees": "Fees",
}
clean_df["target_category"] = clean_df["category"].map(category_mapping)

In [ ]:
for category in ["Subscription", "Travel", "Income", "Transfer"]:
    print(f"\n--- {category} ---")
    examples = clean_df.loc[
        clean_df["category"] == category, "description"
    ].head(10)

    print(examples.to_string(index=False))

In [ ]:
is_debit = clean_df["description"].str.startswith("[debit]",na=False)

is_excluded = clean_df["category"].isin(["Income","Transfer"])
expense_df = clean_df.loc[is_debit & ~is_excluded].copy()


In [ ]:
print("Expense rows:", len(expense_df))

print("\nMapped category counts:")
print(expense_df["target_category"].value_counts())

print("\nSource categories awaiting review:")
print(
    expense_df.loc[
        expense_df["target_category"].isna(), "category"
    ].value_counts()
)

In [ ]:
for category in  ["Subscription","Travel"]:
    descriptions = expense_df.loc[expense_df["category"]==category,"description"]
    print(f"\n---{category}---")
    for description in descriptions.sample(n=20,random_state=42):
        print(description)

In [ ]:
transport_pattern = (
    r"avis|budget rent|dollar rent a car|enterprise|"
    r"hertz|national car rental|alamo|thrifty|turo|"
    r"american airlines|delta air lines|united airlines|"
    r"southwest|jetblue|alaska airlines|spirit airlines|"
    r"frontier airlines|\bairport parking\b"
)

is_travel = expense_df["category"].eq("Travel")

matches_transport = expense_df["description"].str.contains(
    transport_pattern,
    case=False,
    na=False,
    regex=True
)

# Start with Other for Travel rows.
expense_df.loc[is_travel, "target_category"] = "Other"

# Assign Transport where the description matches a transport keyword.
expense_df.loc[
    is_travel & matches_transport,
    "target_category"
] = "Transport"

In [ ]:
for label in ["Transport", "Other"]:
    descriptions = expense_df.loc[
        is_travel & expense_df["target_category"].eq(label),
        "description"
    ]

    print(f"\n--- Travel mapped to {label}: {len(descriptions)} rows ---")

    for description in descriptions.sample(
        n=min(15, len(descriptions)),
        random_state=42
    ):
        print(description)

In [ ]:
is_budget_rental = (
    expense_df["category"].eq("Travel")
    & expense_df["description"].str.contains(
        r"\bbudget\b",
        case=False,
        na=False,
        regex=True
    )
)

expense_df.loc[
    is_budget_rental, "target_category"
] = "Transport"

In [ ]:
for description in expense_df.loc[
    is_budget_rental, "description"
]:
    print(description)

In [ ]:
print(
    expense_df.loc[
        expense_df["category"].eq("Travel"),
        "target_category"
    ].value_counts()
)

In [ ]:
entertainment_pattern = (
    r"netflix|spotify|disney|hulu|hbo|max\.com|"
    r"apple tv|youtube premium|paramount|peacock|"
    r"crunchyroll|audible|playstation|xbox|"
    r"\btubi\b|discord nitro|facebook gaming"
)
education_pattern = (
    r"coursera|udemy|edx|skillshare|"
    r"datacamp|pluralsight|brilliant\.org|duolingo"
)

is_subscription = expense_df["category"].eq("Subscription")

is_entertainment = expense_df["description"].str.contains(
    entertainment_pattern,
    case=False,
    na=False
)


is_education = expense_df["description"].str.contains(
    education_pattern,
    case=False,
    na=False
)

expense_df.loc[is_subscription, "target_category"] = "Other"

expense_df.loc[
    is_subscription & is_entertainment,
    "target_category"
] = "Entertainment"
expense_df.loc[
    is_subscription & is_education,
    "target_category"
] = "Education"

In [ ]:
for label in ["Entertainment", "Education", "Other"]:
    descriptions = expense_df.loc[
        is_subscription
        & expense_df["target_category"].eq(label),
        "description"
    ]

    print(f"\n--- Subscription → {label}: {len(descriptions)} rows ---")

    for description in descriptions.sample(
        n=min(15, len(descriptions)),
        random_state=42
    ):
        print(description)

In [ ]:
other_subscriptions = expense_df.loc[
    is_subscription
    & expense_df["target_category"].eq("Other"),
    "description"
]

for description in other_subscriptions.sample(
    n=min(60, len(other_subscriptions)),
    random_state=17
):
    print(description)
    

In [ ]:
print("Final category counts:")
print(expense_df["target_category"].value_counts())

print("\nRows without a label:")
print(expense_df["target_category"].isna().sum())

print("\nNumber of categories:")
print(expense_df["target_category"].nunique())

In [ ]:
output_path = PROJECT_PATH / "data/raw/processed/expenses_labeled_v1.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

expense_df[
    ["description", "category", "target_category"]
].to_csv(output_path, index=False)

print("Saved to:", output_path.resolve())

In [ ]:
saved_df = pd.read_csv(output_path)

print("Saved rows:", len(saved_df))
print("Matches notebook data:", saved_df.equals(
    expense_df[
        ["description", "category", "target_category"]
    ].reset_index(drop=True)
))

In [ ]:
normalized = (
    expense_df["description"]
    .str.lower()
    .str.replace(r"\d+", " ", regex=True)
    .str.replace(r"[^a-z\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print("Unique original descriptions:")
print(expense_df["description"].nunique())

print("\nUnique normalized descriptions:")
print(normalized.nunique())

print("\nMost frequent normalized descriptions:")
print(normalized.value_counts().head(15))

In [ ]:
label_check = pd.DataFrame({
    "normalized": normalized,
    "target_category": expense_df["target_category"]
})

labels_per_group = (
    label_check.groupby("normalized")["target_category"].nunique()
)

conflicting_groups = labels_per_group[labels_per_group > 1]

print("Normalized groups with conflicting labels:")
print(len(conflicting_groups))

print("\nRows with an empty normalized description:")
print(normalized.eq("").sum())

In [ ]:
conflict_keys = conflicting_groups.index

conflict_rows = expense_df.loc[
    normalized.isin(conflict_keys),
    ["description", "category", "target_category"]
].copy()

conflict_rows["normalized"] = normalized.loc[conflict_rows.index]

for key, group in conflict_rows.groupby("normalized"):
    print(f"\n--- {key} ---")

    print(
        group[
            ["description", "category", "target_category"]
        ]
        .drop_duplicates()
        .head(10)
        .to_string(index=False)
    )

In [ ]:
# Identify rows belonging to conflicting groups.
has_conflict = normalized.isin(conflicting_groups.index)

# Keep them separately for review.
review_df = expense_df.loc[has_conflict].copy()

# Create the dataset we will use for modeling.
model_df = expense_df.loc[~has_conflict].copy()

print("Rows set aside for review:", len(review_df))
print("Rows available for modeling:", len(model_df))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


groups = normalized.loc[model_df.index]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_positions, test_positions = next(
    splitter.split(model_df, groups=groups)
)

train_df = model_df.iloc[train_positions].copy()
test_df = model_df.iloc[test_positions].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

train_groups = set(groups.iloc[train_positions])
test_groups = set(groups.iloc[test_positions])

print("Shared normalized groups:", len(train_groups & test_groups))

print("\nCategory proportions:")
print(pd.DataFrame({
    "train": train_df["target_category"].value_counts(normalize=True),
    "test": test_df["target_category"].value_counts(normalize=True)
}).round(3))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

X_train = train_df["description"]
y_train = train_df["target_category"]

model.fit(X_train, y_train)

print("Training complete.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

X_test = test_df["description"]
y_test = test_df["target_category"]

y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")

print("\nResults by category:")
print(classification_report(
    y_test,  
    y_pred,
    digits=3,
    zero_division=0
))

In [ ]:
results_df = test_df[
    ["description", "target_category"]
].copy()

results_df["predicted_category"] = y_pred

mistakes_df = results_df.loc[
    results_df["target_category"]
    != results_df["predicted_category"]
].copy()

print("Incorrect predictions:", len(mistakes_df))

print("\nMost common confusions:")
print(
    mistakes_df.groupby(
        ["target_category", "predicted_category"]
    ).size().sort_values(ascending=False).head(10)
)

print("\nExample mistakes:")
print(
    mistakes_df.sample(
        n=min(15, len(mistakes_df)),
        random_state=42
    ).to_string(index=False)
)

In [ ]:
import joblib

model_path = PROJECT_PATH / "models/expense_classifier.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model, model_path)
print("Original 12-category baseline saved:", model_path)
